In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.colors as mcolors

from python.benchmark_constants import *

DATA_DIR = Path("./datasets_old")

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.autolayout': True, 'figure.dpi': 120})

In [ ]:
from python.benchmark_io import load_benchmark_data

df_bench = load_benchmark_data(DATA_DIR)

print(f"Loaded {len(df_bench)} individual iterations.")
df_bench.head()

In [ ]:
from python.benchmark_validation import validate_lloyd_parity_parallel

df_parity = validate_lloyd_parity_parallel(data_dir=DATA_DIR, tolerance_pct=1e-5, max_workers=2)
display(df_parity[df_parity["Status"] == "❌ FAIL"])

In [ ]:
from python.benchmark_transforms import *

# Create a master Lloyd-only DataFrame to be used by subsequent charts
df_lloyd = filter_bench(df_bench, phase=PHASE_MAP["lloyd"])

# Mean Python-vs-C++ speedup for each Lloyd configuration
df_summary = compute_language_speedup(df_lloyd)

In [ ]:
from python.benchmark_plotting import *

max_samples = df_bench[COL_SAMPLES].max()

df_math_avg = filter_bench(
    df_bench,
    phase=PHASE_MAP["lloyd"],
    language=LANG_CPP,
    samples=max_samples,
)
df_math_avg[COL_TIME_PER_ITER] = (
    df_math_avg[COL_TIME_S] / df_math_avg[COL_ITERATIONS]
)
df_math_summary = (
    df_math_avg.groupby([COL_CLUSTERS, COL_DIMENSIONS], observed=True)[COL_TIME_PER_ITER]
    .mean()
    .reset_index()
)

df_fixed_avg = mean_time_by_config(
    filter_bench(
        df_bench,
        phase=[PHASE_MAP["soa"], PHASE_MAP["pp"]],
        language=LANG_CPP,
        samples=max_samples,
    ),
    group_cols=[COL_CLUSTERS, COL_DIMENSIONS, COL_PHASE],
)

df_plot = df_fixed_avg.merge(
    df_math_summary,
    on=[COL_CLUSTERS, COL_DIMENSIONS],
)

df_plot[COL_EQUIVALENT_ITERS] = df_plot[COL_TIME_S] / df_plot[COL_TIME_PER_ITER]
df_plot[COL_PHASE] = df_plot[COL_PHASE].cat.remove_unused_categories()

unique_clusters = sorted(df_plot[COL_CLUSTERS].unique())
fig, axes = create_subplot_grid(len(unique_clusters), cols=2, row_height=5, fig_width=12)

for i, cluster in enumerate(unique_clusters):
    ax = axes[i]
    data_cluster = filter_bench(df_plot, clusters=cluster)
    
    # 1. Plot the Relative Cost (Bar chart)
    sns.barplot(
        data=data_cluster,
        x=COL_DIMENSIONS,
        y=COL_EQUIVALENT_ITERS,
        hue=COL_PHASE,
        ax=ax,
        palette={PHASE_MAP["soa"]: "#e74c3c", PHASE_MAP["pp"]: "#f39c12"}
    )
    
    ax.set_title(f"{cluster} Clusters", bbox=small_multiple_title_style)
    
    # Left Axis Styling
    if i % 2 == 0:
        ax.set_ylabel("Cost (Equivalent Lloyd Iters)")
    else:
        ax.set_ylabel("")
        
    ax.set_xlabel("Dimensions")
    ax.grid(axis='y', linestyle='--', alpha=0.6) # Keep only the left grid

    if i != 0: 
        ax.get_legend().remove()
        
    # 2. Plot the Absolute Time (Point plot overlay)
    ax2 = ax.twinx()
    ax2.grid(False)
    
    data_line = data_cluster.drop_duplicates(subset=[COL_DIMENSIONS]).copy()
    data_line[COL_TIME_PER_ITER_MS] = data_line[COL_TIME_PER_ITER] * 1000
    
    sns.pointplot(
        data=data_line,
        x=COL_DIMENSIONS,
        y=COL_TIME_PER_ITER_MS,
        ax=ax2,
        color="black",
        markersize=6,
        errorbar=None
    )
    
    # Right Axis Styling & Color-coding
    if i % 2 == 1:
        ax2.set_ylabel("Absolute Time per Iter (ms)", color="black")
    else:
        ax2.set_ylabel("")

    # Make the right-side spine and ticks match the pointplot color
    ax2.tick_params(axis='y', labelcolor="black")
    ax2.spines['right'].set_color('black')
    ax2.spines['right'].set_linewidth(1.4)

fig.suptitle(
    f"Architectural Overhead: Fixed Costs expressed in Lloyd Iterations\n"
    f"How many iterations worth of time are spent on Setup/Memory? Samples: {format_abbrev(df_bench[COL_SAMPLES].max()):}",
    fontsize=16,
    y=1
)

plt.tight_layout()
plt.show()

In [ ]:
df_math = df_lloyd.copy()

df_math[COL_ITERATION] = df_math.groupby(CONFIG_LANGUAGE_COLS, observed=True).cumcount()

df_pivot = df_math.pivot(
    index=CONFIG_COLS + [COL_ITERATION],
    columns=COL_LANGUAGE,
    values=COL_TIME_S,
).reset_index()

df_pivot = add_speedup(df_pivot)
df_pivot[COL_NBR_CLUSTERS] = df_pivot[COL_CLUSTERS]

g = sns.relplot(
    data=df_pivot,
    x=COL_SAMPLES,
    y=COL_SPEEDUP,
    hue=COL_NBR_CLUSTERS,
    col=COL_DIMENSIONS,     
    col_wrap=3,             
    kind="scatter",
    alpha=0.6,     
    palette="Set1",         
    height=4,
    aspect=1.2,
    facet_kws={'sharey': False} 
)

style_facet_grid(
    g,
    title="Absolute Speedup vs. Sample Size (Lloyd Iterations)",
    title_y=1.01,
    x_log=True,
    sample_x_axis=True,
    titles_inside=True,
    grid_axis="both",
)

g.set_axis_labels("Number of Samples", "Speedup Multiplier (x)")

move_facet_legend_to_row_ends(
    g,
    default_title=COL_NBR_CLUSTERS,
    anchor=(1, 0.5),
)

plt.show()

In [ ]:
# Calculate global max to ensure the color scale is consistent across all subplots
vmax_val = df_summary[COL_SPEEDUP].max()

cluster_vals = sorted(df_summary[COL_CLUSTERS].unique())
fig, axes = create_subplot_grid(len(cluster_vals), cols=2, row_height=6, fig_width=14)

for i, cluster in enumerate(cluster_vals):
    ax = axes[i]
    subset = filter_bench(df_summary, clusters=cluster)

    heatmap_pivot = subset.pivot(
        index=COL_DIMENSIONS,
        columns=COL_SAMPLES,
        values=COL_SPEEDUP,
    )
    heatmap_pivot.columns = [format_abbrev(col) for col in heatmap_pivot.columns]
    
    # We only show the colorbar if it's the second column in the row
    show_cbar = (i % 2 != 0)
    cbar_kwargs = {
        'label': 'Speedup Multiplier (x)', 
        'format': mtick.ScalarFormatter(),
        'ticks': [1, 2, 5, 10, 20, 50, 100]
    } if show_cbar else {}
    
    cbar_ax = ax.inset_axes([1.04, 0, 0.05, 1]) if show_cbar else None
    
    sns.heatmap(
        heatmap_pivot, 
        annot=True, 
        fmt=".2f", 
        cmap="turbo",
        norm=mcolors.LogNorm(vmin=1, vmax=vmax_val),
        vmin=1, 
        vmax=vmax_val,
        ax=ax, 
        cbar=show_cbar,
        cbar_ax=cbar_ax,
        cbar_kws=cbar_kwargs,
        linewidths=.5, 
    )
    
    ax.set_title(f"Clusters: {cluster}", bbox=small_multiple_title_style)
    ax.set_xlabel("Number of Samples")
    ax.set_ylabel("Dimensions" if i % 2 == 0 else "")
    ax.tick_params(axis='x', labelrotation=0)

plt.suptitle(f"Speedup Heatmap ({LANG_CPP} vs. {LANG_PY}) by Number of Clusters\nPhase: {PHASE_MAP['lloyd']}", y=1.0, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare each raw C++ run against the mean Python time for the same config.
# This preserves C++ run-level variance for Seaborn error bands.
df_norm = compute_cpp_speedup_against_python_mean(df_bench)

# 3. Calculate baseline performance retention against the smallest cluster size (Base K)
base_k = df_norm[COL_CLUSTERS].min()

baseline_df = (
    filter_bench(df_norm, clusters=base_k)
    .groupby([COL_PHASE, COL_DIMENSIONS, COL_SAMPLES], observed=True)[COL_SPEEDUP]
    .mean()
    .reset_index()
    .rename(columns={COL_SPEEDUP: COL_BASE_SPEEDUP})
)

df_norm = pd.merge(
    df_norm,
    baseline_df,
    on=[COL_PHASE, COL_DIMENSIONS, COL_SAMPLES],
)

df_norm[COL_RETENTION] = (
    df_norm[COL_SPEEDUP] / df_norm[COL_BASE_SPEEDUP]
) * 100

df_norm[COL_PHASE] = df_norm[COL_PHASE].cat.remove_unused_categories()

for phase in df_norm[COL_PHASE].unique():
    phase_data = filter_bench(df_norm, phase=phase)
    
    g = sns.relplot(
        data=phase_data,
        x=COL_CLUSTERS,
        y=COL_RETENTION,
        hue=COL_SAMPLES,
        col=COL_DIMENSIONS,
        col_wrap=3,
        kind="line",
        errorbar="sd",
        marker="o",
        hue_norm=mcolors.LogNorm(),
        legend="full",
        palette="turbo",
        height=4,  
        aspect=1.2,
        facet_kws={'sharey': True}
    )
    style_facet_grid(
        g,
        title=f"PHASE: {phase.upper()}\nNormalized Efficiency against {base_k} Clusters\nDoes C++ maintain its speed advantage over Python as the Nbr of Clusters increases?",
        title_y=1.06,
        grid_axis="y",
        integer_x_axis=True,
    )

    g.set_axis_labels(COL_NBR_CLUSTERS, "Retention (%)")

    move_facet_legend_to_row_ends(
        g,
        default_title=COL_SAMPLES,
        label_formatter=format_abbrev,
        anchor=(1.05, 0.5),
    )

    for ax in g.axes.flat:
        ax.axhline(100, color="red", linestyle="--", alpha=0.5)

    plt.show()

In [ ]:
median_val = min(
    df_bench[COL_CLUSTERS].unique(),
    key=lambda x: abs(x - df_bench[COL_CLUSTERS].median()),
)

df_line = filter_bench(
    df_bench,
    clusters=median_val,
    language=LANG_CPP,
    phase=PHASE_MAP["lloyd"],
)

df_line[COL_PHASE] = df_line[COL_PHASE].cat.remove_unused_categories()

COL_TIME_PER_ITER_PER_SAMPLE_MS = "Time_per_Lloyd_Iter_per_Sample (ms)"

df_line[COL_TIME_PER_ITER_PER_SAMPLE_MS] = (
    df_line[COL_TIME_S]
    / df_line[COL_SAMPLES]
    / df_line[COL_ITERATIONS]
    * 1000
)

g = sns.relplot(
    data=df_line,
    x=COL_SAMPLES,
    y=COL_TIME_PER_ITER_PER_SAMPLE_MS,
    col=COL_DIMENSIONS,
    col_wrap=3,      
    kind="line",
    errorbar="sd",
    marker="o",
    facet_kws={'sharey': False},
    height=4,
    aspect=1.2,
)

style_facet_grid(
    g,
    title=(
        f"{LANG_CPP} Mean Time per Lloyd Iteration per Sample vs. Sample Size\n"
        f"Does the C++ processing time per sample remain constant as the dataset grows? {COL_NBR_CLUSTERS} = {median_val}"
    ),
    title_y=1.04,
    x_log=True,
    sample_x_axis=True,
)

g.set_axis_labels(
    "Number of Samples",
    "Time per Lloyd Iteration per Sample (ms)",
)

plt.show()